# Debugging a Training Run

Deep learning bugs rarely crash. The code runs, the loss prints, and the model is quietly wrong. This notebook is a toolkit for finding out *why* training misbehaves:

1. **Reproducibility**: seed everything so a bug can be reproduced
2. **Sanity checks**: expected initial loss, then *overfit a single batch*
3. **Common bugs** and the loss-curve signature each one leaves
4. **Weight initialization**: activations that vanish or explode
5. **Gradient inspection**: per-layer norms, clipping, NaN hunting
6. **Learning-rate range test**: choose a learning rate with evidence

All of it runs in seconds on CPU using the 8x8 digits dataset (10 classes).

In [ ]:
import math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler

digits = load_digits()
X = torch.tensor(StandardScaler().fit_transform(digits.data), dtype=torch.float32)
y = torch.tensor(digits.target)
print(X.shape, y.shape)

## 1. Reproducibility

If a run cannot be repeated, a bug cannot be isolated. Seed every random source you use.

In [ ]:
def seed_everything(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)          # also seeds CUDA when present

def make_mlp(depth=3, width=64, act=nn.ReLU, in_f=64, out_f=10):
    layers, f = [], in_f
    for _ in range(depth):
        layers += [nn.Linear(f, width), act()]
        f = width
    layers.append(nn.Linear(f, out_f))
    return nn.Sequential(*layers)

def one_run(seed):
    seed_everything(seed)
    model = make_mlp()
    opt = torch.optim.Adam(model.parameters(), lr=1e-2)
    for _ in range(20):
        opt.zero_grad()
        loss = F.cross_entropy(model(X), y)
        loss.backward()
        opt.step()
    return loss.item()

print('same seed :', one_run(0), one_run(0))
print('other seed:', one_run(1))

Also seed `DataLoader` shuffling (`generator=torch.Generator().manual_seed(seed)`) and worker processes (`worker_init_fn`). For bit-exact GPU runs, add `torch.use_deterministic_algorithms(True)`; it costs speed, so enable it only while debugging.

## 2. Sanity check: the initial loss

For `C` balanced classes, a model that knows nothing predicts `1/C` for each, so cross-entropy should start near `ln(C)`. For 10 classes that is **2.303**. Much higher means a bad init or a bug in the loss; exactly `ln(C)` forever means nothing is learning.

In [ ]:
seed_everything(0)
model = make_mlp()
with torch.no_grad():
    print(f'initial loss: {F.cross_entropy(model(X), y).item():.3f}   expected about ln(10) = {math.log(10):.3f}')

## 3. Overfit one batch

The single most useful debugging step: train on **one small batch** and demand near-zero loss. A correct model with enough capacity must memorise 32 examples. If it cannot, the problem is in the code (loss, labels, optimizer, architecture), not in the data volume or regularization.

Below, a healthy run beside three deliberate bugs. Each leaves a different signature.

In [ ]:
xb, yb = X[:32], y[:32]

def overfit(model, steps=200, lr=1e-2, shuffle_labels=False, loss_fn=F.cross_entropy):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    for _ in range(steps):
        target = yb[torch.randperm(len(yb))] if shuffle_labels else yb
        opt.zero_grad()
        loss = loss_fn(model(xb), target)
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return losses

curves = {}
seed_everything(0); curves['healthy']              = overfit(make_mlp())
seed_everything(0); curves['lr far too high (5.0)'] = overfit(make_mlp(), lr=5.0)
seed_everything(0)
double_softmax = nn.Sequential(make_mlp(), nn.Softmax(dim=1))   # softmax, then CrossEntropy applies another
curves['softmax before CE loss'] = overfit(double_softmax)
seed_everything(0); curves['labels reshuffled each step'] = overfit(make_mlp(), shuffle_labels=True)

for name, c in curves.items():
    print(f'{name:<30} start {c[0]:8.3f}   end {c[-1]:8.3f}   min {min(c):8.3f}')

plt.figure(figsize=(8, 4))
for name, c in curves.items():
    plt.plot(c, label=name)
plt.yscale('log'); plt.xlabel('step'); plt.ylabel('loss (log)'); plt.legend(); plt.title('Overfit one batch')
plt.show()

| Symptom | Likely cause |
|---|---|
| Loss goes to about 0 | healthy |
| Loss stays high and erratic with no steady decline (can also reach `nan`) | learning rate too high. Here Adam at lr=5 hovers near 2.1 instead of exploding; plain SGD at a huge rate is more likely to give `nan`. |
| Loss stalls at about **1.46** | softmax applied before `CrossEntropyLoss`: its logits are capped to [0, 1], so the loss has a floor. `CrossEntropyLoss` and `BCEWithLogitsLoss` expect **raw logits**. |
| Loss stays at about `ln(C)` | the model cannot see a relationship: labels misaligned with inputs, frozen parameters, or `requires_grad=False`. A learning rate that is too high looks similar, so lower it by 10x and rerun to tell them apart. |

Other classics that this check catches: forgetting `optimizer.zero_grad()`, passing the wrong parameters to the optimizer, and detaching the graph by accident.

## 4. Weight initialization

In a deep network every layer multiplies the signal. If each multiplication shrinks the activations a little, a deep stack erases them; if it grows them, they blow up. Initialization chooses the per-layer scale. Below we push random data through a 20-layer stack and track the standard deviation of the activations.

In [ ]:
def activation_stds(act, init, depth=20, width=256):
    seed_everything(0)
    layers = []
    for _ in range(depth):
        lin = nn.Linear(width, width)
        init(lin.weight); nn.init.zeros_(lin.bias)
        layers += [lin, act()]
    net = nn.Sequential(*layers)
    h, stds = torch.randn(1000, width), []
    with torch.no_grad():
        for layer in net:
            h = layer(h)
            if isinstance(layer, act):
                stds.append(h.std().item())
    return stds

inits_tanh = {
    'N(0, 0.01)  too small': lambda w: nn.init.normal_(w, 0, 0.01),
    'N(0, 1)     too large': lambda w: nn.init.normal_(w, 0, 1.0),
    'Xavier':                nn.init.xavier_normal_,
}
inits_relu = {
    'Xavier':  nn.init.xavier_normal_,
    'Kaiming': lambda w: nn.init.kaiming_normal_(w, nonlinearity='relu'),
}

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for name, init in inits_tanh.items():
    s = activation_stds(nn.Tanh, init); ax[0].plot(s, label=name)
    print(f'tanh  {name:<22} std after layer 1: {s[0]:.4f}   after layer 20: {s[-1]:.6f}')
for name, init in inits_relu.items():
    s = activation_stds(nn.ReLU, init); ax[1].plot(s, label=name)
    print(f'relu  {name:<22} std after layer 1: {s[0]:.4f}   after layer 20: {s[-1]:.6f}')
ax[0].set(title='20-layer tanh stack', xlabel='layer', ylabel='activation std', yscale='log'); ax[0].legend()
ax[1].set(title='20-layer ReLU stack', xlabel='layer', yscale='log'); ax[1].legend()
plt.tight_layout(); plt.show()

Reading the plot: with N(0, 0.01) the `tanh` signal dies (std reaches 0). With N(0, 1) the std stays near 1, which looks healthy but is not: `tanh` is pinned at about +1/-1 where its slope is near zero, so gradients vanish (saturation). Xavier shrinks the signal slowly and stays usable. For ReLU, Xavier halves the variance every layer and the signal collapses to about 0.0005, while Kaiming keeps it near 0.5.

**Rule of thumb:** Xavier (Glorot) for `tanh`/`sigmoid`, Kaiming (He) for `ReLU`-family activations. PyTorch's default `nn.Linear` init is a reasonable general choice for shallow nets, which is why earlier notebooks never needed this. It matters as depth grows, and for custom layers you build yourself.

## 5. Gradient inspection

Watching the loss only tells you *that* something is wrong. Per-layer gradient norms tell you *where*.

In [ ]:
def grad_norms(model, xb, yb):
    model.zero_grad()
    F.cross_entropy(model(xb), yb).backward()
    return [m.weight.grad.norm().item() for m in model if isinstance(m, nn.Linear)]

seed_everything(0)
deep_sigmoid = make_mlp(depth=10, width=64, act=nn.Sigmoid)
seed_everything(0)
deep_relu = make_mlp(depth=10, width=64, act=nn.ReLU)
for m in deep_relu:
    if isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, nonlinearity='relu')

g_sig, g_relu = grad_norms(deep_sigmoid, X, y), grad_norms(deep_relu, X, y)
print(f'sigmoid: first layer grad {g_sig[0]:.2e}   last layer grad {g_sig[-1]:.2e}   ratio {g_sig[-1]/g_sig[0]:.0f}x')
print(f'relu:    first layer grad {g_relu[0]:.2e}   last layer grad {g_relu[-1]:.2e}   ratio {g_relu[-1]/g_relu[0]:.1f}x')

plt.figure(figsize=(7, 4))
plt.plot(g_sig, 'o-', label='10 layers, sigmoid'); plt.plot(g_relu, 'o-', label='10 layers, ReLU + Kaiming')
plt.yscale('log'); plt.xlabel('layer (0 = input side)'); plt.ylabel('gradient norm'); plt.legend()
plt.title('Vanishing gradients'); plt.show()

Early layers in the sigmoid net get gradients orders of magnitude smaller than the last layer, so they barely learn. This is the **vanishing gradient** problem, and why ReLU, good init, normalization layers and residual connections exist.

The opposite problem, **exploding gradients**, is handled by clipping the total norm before the optimizer step:

```python
loss.backward()
total_norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # returns the pre-clip norm: log it
optimizer.step()
```

### Hunting NaNs

`torch.autograd.set_detect_anomaly(True)` makes backward raise at the exact operation that produced a `nan`. It is slow, so use it only to locate a bug.

In [ ]:
x = torch.tensor([0.0], requires_grad=True)
try:
    with torch.autograd.detect_anomaly():
        z = torch.sqrt(x) * 0          # d sqrt at 0 is inf, and inf * 0 = nan in backward
        z.backward()
except RuntimeError as e:
    print('caught:', str(e).splitlines()[0])

# cheap guard for training loops
loss = torch.tensor(float('nan'))
print('loss is finite:', bool(torch.isfinite(loss)))

## 6. Learning-rate range test

Instead of guessing, sweep the learning rate upward exponentially over a short run, recording the loss. The best region is where loss falls fastest, and the usable maximum sits just before it blows up. A common pick is about **10x below the loss minimum**.

In [ ]:
def lr_range_test(lr_min=1e-5, lr_max=10, steps=100, batch=64):
    seed_everything(0)
    model = make_mlp()
    opt = torch.optim.SGD(model.parameters(), lr=lr_min, momentum=0.9)
    gamma = (lr_max / lr_min) ** (1 / steps)
    lrs, losses, best = [], [], float('inf')
    for i in range(steps):
        idx = torch.randint(0, len(X), (batch,))
        opt.zero_grad()
        loss = F.cross_entropy(model(X[idx]), y[idx])
        loss.backward(); opt.step()
        lr = opt.param_groups[0]['lr']
        lrs.append(lr); losses.append(loss.item())
        best = min(best, loss.item())
        if loss.item() > 4 * best or not math.isfinite(loss.item()):
            break
        for g in opt.param_groups:
            g['lr'] = lr * gamma
    return lrs, losses

lrs, losses = lr_range_test()
smooth = np.convolve(losses, np.ones(5) / 5, mode='valid')
lrs_s = lrs[2:2 + len(smooth)]
best_lr = lrs_s[int(np.argmin(smooth))]
print(f'lowest smoothed loss at lr = {best_lr:.4f}  ->  suggested lr about {best_lr / 10:.4f}')

plt.figure(figsize=(7, 4))
plt.plot(lrs_s, smooth)
plt.axvline(best_lr / 10, color='g', ls='--', label=f'suggested {best_lr / 10:.4f}')
plt.xscale('log'); plt.xlabel('learning rate'); plt.ylabel('loss'); plt.legend(); plt.title('LR range test (SGD + momentum)')
plt.show()

## 7. Debugging checklist

Work top to bottom; stop at the first failure.

1. **Seed** everything so the failure repeats.
2. **Initial loss** about `ln(C)` (classification) or about the variance of `y` (regression).
3. **Overfit one batch** to about zero loss. If not, fix the code before touching anything else.
4. **Shapes and labels**: print one batch; confirm `x`, `y` dtypes and that labels line up with inputs. For images, plot them.
5. **Logits, not probabilities**, into `CrossEntropyLoss` / `BCEWithLogitsLoss`.
6. **`zero_grad()`, `backward()`, `step()`** in the right order, optimizer holding the right parameters.
7. **`model.train()` / `model.eval()`** set correctly (Dropout and BatchNorm).
8. **Gradient norms** per layer; clip if they explode, rethink init or activations if they vanish.
9. **LR range test** before long runs.
10. Only then scale up data, model size, and regularization.